# 🟢 Lesson 03 — Pandas

**Level: Beginner** · Tables (DataFrames) for assay sheets, geochem databases and catalogues — Excel, but scriptable.

*Part of the companion package for [python_for_geologists](https://github.com/kevinalexandr19/python_for_geologists) by Kevin Alexander Gomez.*

In [1]:
import pandas as pd
from pathlib import Path
DATA = Path("..") / "data"
print("pandas", pd.__version__)

pandas 2.3.3


## 1. Load a real drillhole assay table

In [2]:
assay = pd.read_csv(DATA / "assay.csv")
print("shape:", assay.shape, "| drillholes:", assay["ID"].nunique())
assay.head()

shape: (8332, 11) | drillholes: 55


,ID,FROM,TO,RECOV,CU_pct,AU_gpt,AG_gpt,DENSITY,MO_ppm,AS_ppm,S_pct
0,DH001,0.0,2.0,0.5,0.79,1.75,6.35,NaN,10.0,26.3,0.0
1,DH001,2.0,4.0,1.3,0.83,1.73,5.20,NaN,12.2,31.0,0.0
2,DH001,4.0,6.0,1.8,0.84,6.00,5.75,NaN,24.8,32.5,0.0
3,DH001,6.0,8.0,1.8,0.83,2.56,2.85,2.32,15.7,13.9,0.2
4,DH001,8.0,10.0,2.0,0.97,1.53,2.90,2.98,14.8,15.5,0.5


## 2. Inspect and summarise

In [3]:
assay[["CU_pct", "AU_gpt", "AG_gpt", "MO_ppm"]].describe().round(3)

,CU_pct,AU_gpt,AG_gpt,MO_ppm
count,7951.000,8249.000,7953.000,7953.000
mean,0.952,0.846,3.901,98.888
std,0.916,1.425,15.131,171.304
min,0.000,0.000,0.100,0.200
25%,0.330,0.250,1.350,7.700
50%,0.640,0.510,2.250,30.300
75%,1.290,1.000,3.850,118.400
max,9.290,43.600,505.000,3264.000


## 3. Filter rows — find the high-grade intervals

In [4]:
highgrade = assay[(assay["CU_pct"] >= 1.0) & (assay["AU_gpt"] >= 1.0)]
print(f"{len(highgrade)} of {len(assay)} intervals are >=1% Cu AND >=1 g/t Au")
highgrade.head()

776 of 8332 intervals are >=1% Cu AND >=1 g/t Au


,ID,FROM,TO,RECOV,CU_pct,AU_gpt,AG_gpt,DENSITY,MO_ppm,AS_ppm,S_pct
5,DH001,10.0,12.0,1.90,1.48,2.25,3.35,2.52,39.2,20.2,1.0
6,DH001,12.0,14.0,1.35,1.03,2.24,3.90,2.61,295.0,31.3,1.5
9,DH001,18.0,20.0,2.00,1.66,1.48,3.75,2.96,26.1,9.1,0.8
10,DH001,20.0,22.0,2.00,1.12,2.11,3.30,3.07,16.3,22.5,0.9
12,DH001,24.0,26.0,2.00,1.07,1.55,2.20,3.01,51.7,15.3,0.7


## 4. New columns — interval length and Cu-equivalent grade

In [5]:
assay["LENGTH"] = assay["TO"] - assay["FROM"]
# simple CuEq: Cu% + Au g/t * 0.6 (illustrative conversion factor only)
assay["CUEQ"] = assay["CU_pct"] + assay["AU_gpt"] * 0.6
assay[["ID", "FROM", "TO", "LENGTH", "CU_pct", "AU_gpt", "CUEQ"]].head()

,ID,FROM,TO,LENGTH,CU_pct,AU_gpt,CUEQ
0,DH001,0.0,2.0,2.0,0.79,1.75,1.840
1,DH001,2.0,4.0,2.0,0.83,1.73,1.868
2,DH001,4.0,6.0,2.0,0.84,6.00,4.440
3,DH001,6.0,8.0,2.0,0.83,2.56,2.366
4,DH001,8.0,10.0,2.0,0.97,1.53,1.888


## 5. Group by drillhole — length-weighted mean grade
The single most common resource-geology calculation.

In [6]:
def weighted_cu(g):
    return (g["CU_pct"] * g["LENGTH"]).sum() / g["LENGTH"].sum()

summary = (assay.groupby("ID")
                .apply(weighted_cu, include_groups=False)
                .round(3)
                .rename("CU_wtd_pct")
                .to_frame())
summary["metres"] = assay.groupby("ID")["LENGTH"].sum()
summary.sort_values("CU_wtd_pct", ascending=False)

,CU_wtd_pct,metres
ID,,
DH028,2.864,463.049988
DH023,2.801,233.800003
DH017,2.778,200.000000
DH020,2.045,370.200012
DH016,1.829,200.100006
DH029,1.805,388.299988
DH053,1.745,355.549988
DH012,1.737,602.299988
DH026,1.414,506.700012


## 6. Load the geochem table and cross-tabulate

In [7]:
rocks = pd.read_csv(DATA / "rocks.csv")
print(rocks["Name"].value_counts())
rocks.groupby("Name")[["SiO2", "MgO", "K2O"]].mean().round(2)

Name
basalt      7171
andesite    6271
rhyolite    4968
dacite      4027
Name: count, dtype: int64


,SiO2,MgO,K2O
Name,,,
andesite,57.19,3.87,1.69
basalt,49.31,6.81,1.08
dacite,65.69,1.67,2.59
rhyolite,73.47,0.45,4.04


### ✏️ Try it
1. Compute a length-weighted **Au** grade per drillhole and compare with the Cu ranking.
2. In `rocks`, add a column `K2O_Na2O = K2O / Na2O` and find which rock type has the highest median value.

📚 Docs: https://pandas.pydata.org/docs/